In [13]:
import tensorflow as tf
from tensorflow.keras import layers, models

(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()

x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

train_dataset = (
    tf.data.Dataset.from_tensor_slices((x_train, y_train))
    .shuffle(10000)
    .batch(64)
)

test_dataset = (
    tf.data.Dataset.from_tensor_slices((x_test, y_test))
    .batch(64)
)

In [14]:
model = models.Sequential([
    layers.Flatten(input_shape=(28, 28)),
    layers.Dense(128, activation="relu"),
    layers.Dense(64, activation="relu"),
    layers.Dense(10, activation="softmax")
])

/usr/local/lib/python3.13/dist-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [15]:
loss_function = tf.keras.losses.SparseCategoricalCrossentropy()

optimizer = tf.keras.optimizers.Adam()

train_accuracy = tf.keras.metrics.SparseCategoricalAccuracy()
test_accuracy = tf.keras.metrics.SparseCategoricalAccuracy()

In [16]:
@tf.function
def train_step(images, labels):

    with tf.GradientTape() as tape:

        predictions = model(images, training=True)

        loss = loss_function(labels, predictions)

    gradients = tape.gradient(
        loss,
        model.trainable_variables
    )

    optimizer.apply_gradients(
        zip(gradients, model.trainable_variables)
    )

    train_accuracy.update_state(labels, predictions)

    return loss

In [17]:
@tf.function
def test_step(images, labels):

    predictions = model(images, training=False)

    loss = loss_function(labels, predictions)

    test_accuracy.update_state(labels, predictions)

    return loss

In [18]:
epochs = 5

for epoch in range(epochs):

    train_accuracy.reset_state()
    test_accuracy.reset_state()

    train_losses = []
    test_losses = []

    # Training
    for images, labels in train_dataset:
        loss = train_step(images, labels)
        train_losses.append(loss)

    # Testing
    for images, labels in test_dataset:
        loss = test_step(images, labels)
        test_losses.append(loss)

    avg_train_loss = tf.reduce_mean(train_losses)
    avg_test_loss = tf.reduce_mean(test_losses)

    print(
        f"Epoch {epoch + 1}/{epochs} | "
        f"Train Loss: {avg_train_loss:.4f} | "
        f"Train Accuracy: {train_accuracy.result():.4f} | "
        f"Test Loss: {avg_test_loss:.4f} | "
        f"Test Accuracy: {test_accuracy.result():.4f}"
    )

Epoch 1/5 | Train Loss: 0.2802 | Train Accuracy: 0.9194 | Test Loss: 0.1505 | Test Accuracy: 0.9557
Epoch 2/5 | Train Loss: 0.1134 | Train Accuracy: 0.9656 | Test Loss: 0.0996 | Test Accuracy: 0.9703
Epoch 3/5 | Train Loss: 0.0775 | Train Accuracy: 0.9757 | Test Loss: 0.1004 | Test Accuracy: 0.9687
Epoch 4/5 | Train Loss: 0.0591 | Train Accuracy: 0.9816 | Test Loss: 0.0872 | Test Accuracy: 0.9726
Epoch 5/5 | Train Loss: 0.0468 | Train Accuracy: 0.9845 | Test Loss: 0.0881 | Test Accuracy: 0.9723
